# Topic 3.4 - 子查询

## 1. 子查询 - FROM 子句中的子查询

子查询是 SQL 中的一种语法结构，和 CTE 类似，子查询也是 SQL 里面套 SQL：

- 具体来说，子查询分为三种类型:

    - `FROM` 子句中的子查询
    - `SELECT` 子句中的子查询
    - `WHERE` 子句中的子查询
 - 我们先来讲一下 `FROM` 子句中的子查询，因为它和我们上一节讲的 CTE 的概念最为接近

`FROM` 子句中的子查询和 CTE 的概念非常相似，都是在一个 SQL 的查询结果上继续查询数据，它们的语法区别如下：

- CTE 的语法是：

```sql
WITH
临时表1 AS (
    SELECT ...
),
临时表2 AS (
    SELECT ...
),
...
临时表N AS (
    SELECT ...
)
SELECT ...
FROM 临时表1
JOIN 临时表2 ON ...
JOIN ... ON ...
```

- `FROM` 子句中的子查询的语法是：

```sql
SELECT ...
FROM (
    SELECT ...
) AS 子查询表1
JOIN (
    SELECT ...
) AS 子查询表2 ON 子查询表1.连接键 = 子查询表2.连接键
JOIN (
    SELECT ...
) AS 子查询表3 ON 子查询表1.连接键 = 子查询表3.连接键
...
```

子查询的特征是：

- 支持一个查询中定义多个子查询，每个子查询都是一个完整的 SQL 语句，可以包含各种子句，例如 `WHERE`、`GROUP BY`、`ORDER BY`、`JOIN`、`UNION` 等等，甚至可以嵌套子查询
- 但是，**子查询之间不可以相互引用**，也就是说，在定义子查询的时候，不能使用之前定义的子查询来查询数据，哪怕我们给子查询起了别名也不行
- 子查询只能通过 JOIN 的方式来连接在一起，不能使用 UNION 来连接在一起，否则会报错

我们来简单对比一下 CTE 和 `FROM` 子句中的子查询：

- CTE 不支持 CTE 嵌套，但是子查询是支持子查询嵌套的
- CTE 支持 CTE 之间相互引用，但是子查询不支持子查询之间相互引用
- CTE 支持 CTE 之间通过 JOIN 或者 UNION 来连接在一起，但是子查询只能通过 JOIN 来连接在一起，不能使用 UNION 来连接在一起

但是大家会发现，子查询远不如 CTE 灵活好用

- 一方面是，上面我们对比过 CTE 和子查询，CTE 的功能更强大，支持 CTE 之间相互引用，支持 CTE 之间通过 JOIN 或者 UNION 来连接在一起，而子查询不支持这些功能
- 另一方面的原因是，CTE 是先准备子表，再进行查询，而 `FROM` 子句中的子查询缺正好相反，得先想好自己要查询什么样的数据，才去写子查询，这个思维是有点反人类的
- 因此，如果没有专门要求，我们更推荐大家使用 CTE，来实现从一个查询结果上继续查询数据的功能



这里我们来看一个例子，使用 CTE 和 `FROM` 子句中的子查询，来实现同样的功能：

- 查询出消费金额大于所在国家平均消费金额的客户的 CustomerId、FirstName、LastName、Country、总消费金额，并按总消费金额降序排列：

- 使用 CTE 来实现：

In [2]:
%%sql
WITH
CustomerSpending AS (
    SELECT
        C.CustomerId,
        C.FirstName,
        C.LastName,
        C.Country,
        I.Total AS TotalSpending
    FROM Customer AS C
    JOIN Invoice  AS I ON C.CustomerId = I.CustomerId
),
CountryAverageSpending AS (
    SELECT
        Country,
        AVG(TotalSpending) AS AverageSpending
    FROM CustomerSpending
    GROUP BY Country
)
SELECT
    CS.CustomerId,
    CS.FirstName,
    CS.LastName,
    CS.Country,
    CS.TotalSpending,
    CAS.AverageSpending
FROM CustomerSpending       AS CS
JOIN CountryAverageSpending AS CAS ON CS.Country = CAS.Country
WHERE CS.TotalSpending > CAS.AverageSpending
ORDER BY CS.TotalSpending DESC

,CustomerId,FirstName,LastName,Country,TotalSpending,AverageSpending
0,6,Helena,Holý,Czech Republic,25.86,6.445714
1,26,Richard,Cunningham,USA,23.86,5.747912
2,45,Ladislav,Kovács,Hungary,21.86,6.517142
3,46,Hugh,O'Reilly,Ireland,21.86,6.517142
4,7,Astrid,Gruber,Austria,18.86,6.088571
...,...,...,...,...,...,...
167,52,Emma,Jones,United Kingdom,5.94,5.374285
168,53,Phil,Hughes,United Kingdom,5.94,5.374285
169,54,Steve,Murray,United Kingdom,5.94,5.374285
170,55,Mark,Taylor,Australia,5.94,5.374285


- 使用 `FROM` 子句中的子查询来实现：

In [5]:
%%sql
SELECT
    CS.CustomerId,
    CS.FirstName,
    CS.LastName,
    CS.Country,
    CS.TotalSpending,
    CAS.AverageSpending
FROM (
    SELECT
        C.CustomerId,
        C.FirstName,
        C.LastName,
        C.Country,
        I.Total AS TotalSpending
    FROM Customer AS C
    JOIN Invoice  AS I ON C.CustomerId = I.CustomerId
) AS CS
JOIN (
    SELECT
    Country,
    AVG(TotalSpending) AS AverageSpending
    FROM (
        SELECT
            C.CustomerId,
            C.FirstName,
            C.LastName,
            C.Country,
            I.Total AS TotalSpending
        FROM Customer AS C
        JOIN Invoice  AS I ON C.CustomerId = I.CustomerId
    ) AS CS2
    GROUP BY Country
) AS CAS ON CS.Country = CAS.Country
WHERE CS.TotalSpending > CAS.AverageSpending
ORDER BY CS.TotalSpending DESC

,CustomerId,FirstName,LastName,Country,TotalSpending,AverageSpending
0,6,Helena,Holý,Czech Republic,25.86,6.445714
1,26,Richard,Cunningham,USA,23.86,5.747912
2,45,Ladislav,Kovács,Hungary,21.86,6.517142
3,46,Hugh,O'Reilly,Ireland,21.86,6.517142
4,7,Astrid,Gruber,Austria,18.86,6.088571
...,...,...,...,...,...,...
167,52,Emma,Jones,United Kingdom,5.94,5.374285
168,53,Phil,Hughes,United Kingdom,5.94,5.374285
169,54,Steve,Murray,United Kingdom,5.94,5.374285
170,55,Mark,Taylor,Australia,5.94,5.374285


## 2. 子查询 - SELECT 子句中的子查询

我们之前介绍过，在 `SELECT` 中，如果 `SELECT` 一个常数，那么这个常数的值在查询结果中所有行都是一样的，比方说：

In [6]:
%%sql
SELECT
    EmployeeId,
    FirstName,
    LastName,
    1 AS ConstantValue
FROM Employee

,EmployeeId,FirstName,LastName,ConstantValue
0,1,Andrew,Adams,1
1,2,Nancy,Edwards,1
2,3,Jane,Peacock,1
3,4,Margaret,Park,1
4,5,Steve,Johnson,1
5,6,Michael,Mitchell,1
6,7,Robert,King,1
7,8,Laura,Callahan,1


但有的时候，我们并不想手写一个常数，而是从数据库中查出一个常数，这个时候就可以使用 `SELECT` 子句中的子查询来实现：

- 这个子查询结果必须是一个单值，也就是说，这个子查询必须只能返回一行一列的数据，否则会报错
- 如果子查询结果，是一个一列多行，或一行多列，或者多行多列的数据，那么在 `SELECT` 子句中的子查询就无法使用了，会报错
- 因为 SQL 并不知道要把这个多值的结果放在查询结果中的哪一行哪一列，所以就会报错

我们来看一个简单的例子：

In [15]:
%%sql
SELECT
    EmployeeId,
    FirstName,
    LastName,
    (SELECT COUNT(*) FROM Employee WHERE ReportsTo IS NOT NULL) AS EmployeeCount
FROM Employee

,EmployeeId,FirstName,LastName,EmployeeCount
0,1,Andrew,Adams,7
1,2,Nancy,Edwards,7
2,3,Jane,Peacock,7
3,4,Margaret,Park,7
4,5,Steve,Johnson,7
5,6,Michael,Mitchell,7
6,7,Robert,King,7
7,8,Laura,Callahan,7


## 3. 子查询 - WHERE 子句中的子查询

在 `WHERE` 子句中使用子查询也是非常常见的，它其实就是我们之前讲过的 `WHERE` 的两个场景的延伸：

- 在 `WHERE` 字句中，判断一个列的值是否是一个特定值：`WHERE 列名 = 常数`，这里这个常数可以是子查询查出来的单值结果
- 在 `WHERE` 字句中，判断一个列的值是否在一个特定的值的集合中：`WHERE 列名 IN (常数1, 常数2, ...)`，这里这个常数集合也可以是子查询查出来的多值结果

### (1) 判断单值的情况

我们先来看一下第一个场景，在 `WHERE` 字句中，判断一个列的值是否是一个特定值：

- 这时的子查询语法格式是 `WHERE 列名 = (SELECT ...)`，这里的子查询必须只能返回一个单值结果，否则会报错
- 例如，我们想查询销量最高的 Track 以及其 Artist 信息，这时我们就可以使用 `WHERE` 子句中的子查询来实现：
- 首先，我们先把子查询单独写出来，来查询销量最高的 TrackId：

In [12]:
%%sql
SELECT TOP 1
    TrackId
FROM InvoiceLine
GROUP BY TrackId
ORDER BY SUM(Quantity) DESC

,TrackId
0,2


- 如果按照我们之前的思路来查询销量最高的 Track 以及其 Artist 信息，我们会直接记下来这个 TrackId 的值，也就是 `2`，然后写一个 SQL 语句来查询这个 TrackId 的 Track 信息和 Artist 信息：

```sql
SELECT
    T.TrackId,
    T.Name     AS TrackName,
    T.AlbumId,
    A.ArtistId,
    A.Name     AS ArtistName
FROM Track  AS T
JOIN Album  AS B ON T.AlbumId  = B.AlbumId
JOIN Artist AS A ON B.ArtistId = A.ArtistId
WHERE TrackId = 2
```

- 但是子查询允许我们直接把查询销量最高的 TrackId 的子查询放在 `WHERE` 子句中，这样我们就不需要事先知道这个 TrackId 的值了：

In [10]:
%%sql
SELECT
    T.TrackId,
    T.Name     AS TrackName,
    T.AlbumId,
    A.ArtistId,
    A.Name     AS ArtistName
FROM Track  AS T
JOIN Album  AS B ON T.AlbumId  = B.AlbumId
JOIN Artist AS A ON B.ArtistId = A.ArtistId
WHERE TrackId = (
    SELECT TOP 1
        TrackId
    FROM InvoiceLine
    GROUP BY TrackId
    ORDER BY SUM(Quantity) DESC
)

,TrackId,TrackName,AlbumId,ArtistId,ArtistName
0,2,Balls to the Wall,2,2,Accept


### (2) 判断多值的情况

第二个场景，也就是在 `WHERE` 字句中，判断一个列的值是否在一个特定的值的集合中：

- 这时的子查询语法格式是 `WHERE 列名 IN (SELECT ...)`，这里的子查询必须只能返回一个一列多行的结果，否则会报错
- 例如，我们想查询销量前10的 Track 以及它们所属的 Artist 的信息，这时我们就可以使用 `WHERE` 子句中的子查询来实现：
- 首先，同样的思路，我们先把子查询单独写出来，来查询销量前10的 TrackId：

In [15]:
%%sql
SELECT TOP 10
    TrackId
FROM InvoiceLine
GROUP BY TrackId
ORDER BY SUM(Quantity) DESC

,TrackId
0,9
1,8
2,2
3,20
4,32
5,48
6,66
7,84
8,161
9,162


- 按照我们之前的思路来查询销量前10的 Track 以及它们所属的 Artist 的信息，我们会直接记下来这10个 TrackId 的值，然后写一个 SQL 语句来查询这些 TrackId 的 Track 信息和 Artist 信息：

```sql
SELECT
    T.TrackId,
    T.Name     AS TrackName,
    T.AlbumId,
    A.ArtistId,
    A.Name     AS ArtistName
FROM Track  AS T
JOIN Album  AS B ON T.AlbumId  = B.AlbumId
JOIN Artist AS A ON B.ArtistId = A.ArtistId
WHERE TrackId IN (9, 8, 2, 20, 32, 48, 66, 84, 161, 162)
```

- 但是子查询允许我们直接把查询销量前10的 TrackId 的子查询放在 `WHERE` 子句中，这样我们就不需要事先知道这10个 TrackId 的值了：

In [16]:
%%sql
SELECT
    T.TrackId,
    T.Name     AS TrackName,
    T.AlbumId,
    A.ArtistId,
    A.Name     AS ArtistName
FROM Track  AS T
JOIN Album  AS B ON T.AlbumId  = B.AlbumId
JOIN Artist AS A ON B.ArtistId = A.ArtistId
WHERE TrackId IN (
    SELECT TOP 10
        TrackId
    FROM InvoiceLine
    GROUP BY TrackId
    ORDER BY SUM(Quantity) DESC
)

,TrackId,TrackName,AlbumId,ArtistId,ArtistName
0,9,Snowballed,1,1,AC/DC
1,8,Inject The Venom,1,1,AC/DC
2,2,Balls to the Wall,2,2,Accept
3,20,Overdose,4,1,AC/DC
4,32,Deuces Are Wild,5,3,Aerosmith
5,48,Not The Doctor,6,4,Alanis Morissette
6,66,Por Causa De Você,8,6,Antônio Carlos Jobim
7,84,Welcome Home (Sanitarium),9,7,Apocalyptica
8,161,Snowblind,17,12,Black Sabbath
9,162,Cornucopia,17,12,Black Sabbath


### (3) WHERE 子句中使用 EXISTS 关键字

在 `WHERE` 子句中，判断一个列的值是否在一个特定的值的集合中，除了使用 `IN` 关键字以外，还可以使用 `EXISTS` 关键字来实现：

- `EXISTS` 关键字的语法格式是 `WHERE EXISTS (SELECT ...)`，这里的子查询必须只能返回一个一列多行的结果，否则会报错
- `EXISTS` 关键字和 `IN` 关键字的区别在于：

    - `IN` 关键字会把子查询返回的结果作为一个值的集合来进行判断，
    - `EXISTS` 关键字则是判断子查询是否有结果返回（子查询结果行数是否大于1），如果有结果返回，那么 `EXISTS` 就会返回 `TRUE`，如果没有结果返回，那么 `EXISTS` 就会返回 `FALSE`
    - 也就是说，`EXISTS` 关键字只关心有无查询结果，不关心查询结果是什么

- 因此，在使用 `EXISTS` 关键字时，子查询中通常会包含一个连接条件来连接外层查询和子查询，这样才能保证子查询的结果是有意义的，否则 `EXISTS` 关键字就没有什么作用了

在 SQL 中，`EXISTS` 关键字的语法有一点特殊，我们来通过一个例子说明一下：

- 下面这段代码会查询出所有曾经被卖出过的 Track 的 TrackId 和 Name：

In [20]:
%%sql
SELECT DISTINCT
    T.TrackId,
    T.Name
FROM Track AS T
WHERE EXISTS (
    SELECT 1
    FROM InvoiceLine AS IL
    WHERE IL.TrackId = T.TrackId
)

,TrackId,Name
0,1,For Those About To Rock (We Salute You)
1,2,Balls to the Wall
2,3,Fast As a Shark
3,4,Restless and Wild
4,5,Princess of the Dawn
...,...,...
1979,3493,"Metopes, Op. 29: Calypso"
1980,3494,"Symphony No. 2, Op. 16 - ""The Four Temperamen..."
1981,3496,"Étude 1, In C Major - Preludio (Presto) - Liszt"
1982,3499,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...


- 在这段代码中，子查询中的 `WHERE IL.TrackId = T.TrackId` 规定了匹配条件
- 在执行外层查询时，SQL 会检查 `Track` 表里的每一行 `TrackId`，看一下这个 `TrackId` 是否在 `InvoiceLine` 表中的 `TrackId` 列中出现过
- 如果出现过，那么这个 `TrackId` 就满足 `EXISTS` 的条件，就会被返回到查询结果中，如果没有出现过，那么这个 `TrackId` 就不满足 `EXISTS` 的条件，就不会被返回到查询结果中
- 子查询里的 `SELECT 1` 是什么含义呢，其实是因为 `EXISTS` 关键字只关心子查询是否有结果返回，而不关心子查询返回的结果是什么，所以我们在子查询中随便写一个常数 `1` 就可以了，写成 `SELECT 1`、`SELECT *`、`SELECT 列1, 列2` 都是一样的
- 也就是说，此时，`WHERE IL.TrackId = T.TrackId` 这个条件才是关键，`SELECT` 里面写什么无所谓

我们再来看一个例子：

- 查询出所有买过 `Rock` 风格的专辑的客户的 CustomerId、FirstName、LastName：

In [29]:
%%sql
SELECT
    C.CustomerId,
    C.FirstName,
    C.LastName
FROM Customer AS C
WHERE EXISTS (
    SELECT 1
    FROM Invoice AS I
    JOIN InvoiceLine AS IL ON I.InvoiceId = IL.InvoiceId
    JOIN Track AS T ON IL.TrackId = T.TrackId
    JOIN Album AS A ON T.AlbumId = A.AlbumId
    JOIN Genre AS G ON T.GenreId = G.GenreId
    WHERE I.CustomerId = C.CustomerId
    AND G.Name LIKE '%Rock%'
)

,CustomerId,FirstName,LastName
0,1,Luís,Gonçalves
1,2,Leonie,Köhler
2,3,François,Tremblay
3,4,Bjørn,Hansen
4,5,František,Wichterlová
5,6,Helena,Holý
6,7,Astrid,Gruber
7,8,Daan,Peeters
8,9,Kara,Nielsen
9,10,Eduardo,Martins


- 在这段代码的子查询中，`WHERE I.CustomerId = C.CustomerId` 规定了匹配条件，`AND G.Name LIKE '%Rock%'` 规定了过滤条件
- 在执行外层查询时，SQL 会检查 `Customer` 表里的每一行 `CustomerId`，看一下这个 `CustomerId` 是否在子查询中满足条件 `I.CustomerId = C.CustomerId AND G.Name LIKE '%Rock%'` 的行中出现过
- 如果出现过，那么这个 `CustomerId` 就满足 `EXISTS` 的条件，就会被返回到查询结果中，如果没有出现过，那么这个 `CustomerId` 就不满足 `EXISTS` 的条件，就不会被返回到查询结果中